In [4]:
from cube_nn import CubeValueResNet
import torch
from cube import Cube
from solvers import SuperSolver
from cube_nn import NNValueFunctionType
from tqdm import tqdm
import time

In [5]:
# Load NN value function
I = 500
net = CubeValueResNet()
net.load_state_dict(torch.load(f'temp_models/resnet2.2/cube_value_resnet_iter_{I}.pth'))

# Move model to CUDA if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = net.to(device)
print(f'Using device: {device}')

# Use standard value function type
net.set_value_function_type(NNValueFunctionType.STANDARD)

Using device: cuda


/tmp/ipykernel_287590/2195543875.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(f'temp_models/resnet2.2/cube_value_resnet_iter_{I}.pth'))

In [6]:

solver = SuperSolver(net.as_value_function(), weight=0.25, noise=0.0,
                            max_moves=40, max_queue_size=1000000, t_max=60, max_restarts=2, batch_size=25, seed=42, dataset_moves=5)

Full optimal value dataset not found at datasets/full/full_5.pt, generating it...
Generating cubes with 1 moves...
Added 18 new cubes.
Generating cubes with 2 moves...
Added 243 new cubes.
Generating cubes with 3 moves...
Added 3240 new cubes.
Generating cubes with 4 moves...
Added 43239 new cubes.
Generating cubes with 5 moves...
Added 574908 new cubes.
Saved full optimal value dataset to datasets/full/full_5.pt.


In [7]:
# Generate cubes to evaluate the solver on
N = 100
scramble_moves = 50
cubes = [Cube(n_scramble_moves=scramble_moves, scramble_seed=i) for i in range(N)]

In [8]:
solved_cubes = 0
unsolved_cubes = 0
sol_lengths = []
sol_times = []
for cube in tqdm(cubes, desc=f'Evaluating on cubes', leave=False):
    start_time = time.time()
    solution = solver(cube)
    end_time = time.time()
    if solution is not None:
        solved_cubes += 1
        sol_lengths.append(len(solution))
        sol_times.append(end_time - start_time)
    else:
        unsolved_cubes += 1
        print('Unsolved cubes: ', unsolved_cubes)
print(f'Solved: {solved_cubes}/{N}')
print(f'Fraction solved: {solved_cubes / N}')
print(f'Average solution length: {sum(sol_lengths) / len(sol_lengths) if sol_lengths else 0}')
print(f'Average solution time: {sum(sol_times) / len(sol_times) if sol_times else 0}')

Solved: 100/100
Fraction solved: 1.0
Average solution length: 27.94
Average solution time: 7.388394718170166
